In [ ]:
from glob import glob
import os
import matplotlib.pyplot as plt
plt.rcParams['figure.constrained_layout.use'] = True
import numpy as np
import pandas as pd
from tqdm import tqdm
import shutil

In [ ]:
base_dir = "../../data/test_alignment_output"
methods = ["random", "lengthInform", "mlmcal"]
toppercent = 0.01  # top 1%

Seg_nums = [4, 8, 12]
reference_method = {"4": "mlmcal_top11_gamma0.1_a0.25", "8": "mlmcal_top6_gamma0.1_a0.25", "12": "mlmcal_top4_gamma0.1_a0.25"}

In [3]:
for method in methods:

    for Seg_num in Seg_nums:

        test_truth_dir = f"{base_dir}/seg{Seg_num}_{reference_method[str(Seg_num)]}"
    
        if method != "mlmcal":
            test_method_dir = f"{base_dir}/seg{Seg_num}_{method}_toppercent{toppercent}/"

            truth_segs = glob(os.path.join(test_truth_dir, "*_truth_*"))
            for seg in tqdm(truth_segs):
                shutil.copy(seg, test_method_dir)

  0%|          | 0/1144 [00:00<?, ?it/s]

100%|██████████| 406/406 [00:00<00:00, 2017.20it/s]


In [ ]:
######### for extact top-1 alignment overlap ratio ##########
for method in methods:
    
    print(f"================ Method: {method} ==================")

    for Seg_num in Seg_nums:

        test_truth_dir = f"{base_dir}/seg{Seg_num}_{reference_method[str(Seg_num)]}"
        if method == "mlmcal":
            test_method_dir = test_truth_dir
        else:
            test_method_dir = f"{base_dir}/seg{Seg_num}_{method}_toppercent{toppercent}/"

        truth_segs = glob(os.path.join(test_method_dir, "*_truth_*_sylphones.txt"))

        matched_seg = []
        num_percentk = round(len(truth_segs) * 2 * toppercent)

        for seg in truth_segs:
            seg_id = seg.split('/')[-1].split('_truth_')[0]
            # if exist path with same seg_id in submitted dir, then compare sylphones
            if num_percentk < 10:
                top_match = glob(os.path.join(test_method_dir, seg_id+f'_[0-9]_'+seg_id+'_sylphones.txt'))
            else:

                top_match = glob(os.path.join(test_method_dir, seg_id+f'_[0-{int(str(num_percentk)[0])}][0-{int(str(num_percentk)[1])}]_'+seg_id+'_sylphones.txt')) + \
                            glob(os.path.join(test_method_dir, seg_id+f'_[0-9]_'+seg_id+'_sylphones.txt'))
                            
            if top_match:
                matched_seg.append((seg_id, top_match[0]))

        # alignment distance
        ratio_overlap = []
        for seg_id, seg in matched_seg:
            pred_align = pd.read_csv(seg, sep='\t', header=None)
            truth_align = pd.read_csv(os.path.join(test_method_dir, f'{seg_id}_truth_{seg_id}_sylphones.txt'), sep='\t', header=None)

            cols = truth_align.columns.intersection(pred_align.columns)

            h1 = pd.util.hash_pandas_object(truth_align[cols], index=False)
            h2 = pd.util.hash_pandas_object(pred_align[cols], index=False)

            n_overlap = h1.drop_duplicates().isin(h2.drop_duplicates()).sum()

            seg_ratio_overlap = n_overlap / len(truth_align)
            ratio_overlap.append(seg_ratio_overlap)

        # average overlap ratio
        avg_overlap = np.mean(ratio_overlap)
        print("----------------------------------")
        print(f"Total number of segments: {len(truth_segs)}, {toppercent*100:.0f}% = {num_percentk}, get {len(matched_seg)}")
        print(f"Seg {Seg_num}: average overlap ratio: {avg_overlap:.4f}")

================ Method: random ==================


/home/ids/chawang/miniconda3/envs/mlm_new/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/ids/chawang/miniconda3/envs/mlm_new/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


----------------------------------
Total number of segments: 572, 1% = 11, get 0
Seg 4: average overlap ratio: nan
----------------------------------
Total number of segments: 308, 1% = 6, get 2
Seg 8: average overlap ratio: 0.5897
----------------------------------
Total number of segments: 203, 1% = 4, get 3
Seg 12: average overlap ratio: 0.2692
================ Method: lengthInform ==================
----------------------------------
Total number of segments: 572, 1% = 11, get 12
Seg 4: average overlap ratio: 0.9575
----------------------------------
Total number of segments: 308, 1% = 6, get 11
Seg 8: average overlap ratio: 0.8564
----------------------------------
Total number of segments: 203, 1% = 4, get 8
Seg 12: average overlap ratio: 0.8301
================ Method: mlmcal ==================
----------------------------------
Total number of segments: 572, 1% = 11, get 235
Seg 4: average overlap ratio: 0.8721
----------------------------------
Total number of segments: 308, 1

In [25]:
# check ratio of perturbed lyrics in top-1% returned by MLM-CAL

for seg_num in Seg_nums:
    mlm_cal_dir = os.path.join(base_dir, "seg" + str(seg_num) + "_" + reference_method[str(seg_num)])
    seg_files = glob(os.path.join(mlm_cal_dir, "*_truth_*_words.txt"))

    top1percent = round(len(seg_files) * 2 * 0.01)
    seg_ids = [os.path.basename(file).split("_truth_")[0] for file in seg_files]

    perturb_ratio = []
    for seg in seg_ids:
        perturb_num = len(glob(os.path.join(mlm_cal_dir, seg + "*_shuffled_words.txt")))
        perturb_ratio.append(perturb_num / top1percent)

    perturb_ratio = np.array(perturb_ratio)
    print(f"Ratio of perturned lyrics within top-1% for Seg {seg_num}: {perturb_ratio.mean()}")

Ratio of perturned lyrics within top-1% for Seg 4: 0.3329624920534011
Ratio of perturned lyrics within top-1% for Seg 8: 0.2483766233766234
Ratio of perturned lyrics within top-1% for Seg 12: 0.17610837438423646
